[NOTEBOOK LINK](https://colab.research.google.com/drive/1ikUIPXguuxRQR7qTxpZAgVVi20U4AuVW)

In [ ]:
# Installing and loading sqldf for running SQL queries on R dataframes
install.packages("sqldf")
library(sqldf)

print("packages loaded successfully")


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite



[1] "packages loaded successfully"


In [ ]:
# Loading cleaned datasets from GitHub
# Using cleaned versions to avoid repeating the cleaning process
BASE_URL = "https://raw.githubusercontent.com/DaNIsH-768/DBA_ASSESSMENT/refs/heads/main/cleaned_data/"

complaints <- read.csv(paste0(BASE_URL, "complaints_clean.csv"))
deliveries <- read.csv(paste0(BASE_URL, "deliveries_clean.csv"))
orders     <- read.csv(paste0(BASE_URL, "orders_clean.csv"))
customers  <- read.csv(paste0(BASE_URL, "customers_clean.csv"))
drivers    <- read.csv(paste0(BASE_URL, "drivers_clean.csv"))
vehicles   <- read.csv(paste0(BASE_URL, "vehicles_clean.csv"))
hubs       <- read.csv(paste0(BASE_URL, "hubs_clean.csv"))


In [ ]:
# Quick inspection of deliveries - the central table
head(deliveries)
print("-----------------------------------")
dim(deliveries)
print("-----------------------------------")
summary(deliveries)


,delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost,duration_hours,complaint_filed,on_time_flag,failure_flag
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>,<int>,<int>,<int>
1,DL00001,O00938,D004,V056,H05,2024-06-18 10:57:00,2024-06-19 09:05:59.904311,Failed,17.26,1,0,3.07,12.05,22.149973,0,0,1
2,DL00002,O00004,D138,V007,H02,2025-01-11 18:45:00,2025-01-11 17:39:00.000000,OnTime,10.34,1,0,5.00,13.41,1.100000,0,1,0
3,DL00003,O00639,D006,V049,H02,2025-06-02 20:39:00,2025-06-02 21:45:32.366770,OnTime,7.92,0,0,4.98,8.51,1.108991,0,1,0
4,DL00004,O00313,D116,V055,H02,2024-03-08 23:31:00,2024-03-09 23:30:08.103702,Delayed,16.42,0,0,4.18,13.62,23.985584,0,0,0
5,DL00005,O00844,D108,V034,H01,2025-09-21 11:43:00,2025-09-21 15:45:34.131056,OnTime,14.52,1,0,4.18,9.22,4.042814,0,1,0
6,DL00006,O00029,D037,V098,H03,2024-09-11 12:40:00,2024-09-12 17:11:52.384869,Delayed,13.84,0,0,1.57,9.58,28.531218,0,0,0


[1] "-----------------------------------"


[1] 950  17

[1] "-----------------------------------"


    delivery_id       order_id       driver_id       vehicle_id 
 Length   :950   Length   :950   Length   :950   Length   :950  
 N.unique :950   N.unique :950   N.unique :170   N.unique :120  
 N.blank  :  0   N.blank  :  0   N.blank  :  0   N.blank  :  0  
 Min.nchar:  7   Min.nchar:  6   Min.nchar:  4   Min.nchar:  4  
 Max.nchar:  7   Max.nchar:  6   Max.nchar:  4   Max.nchar:  4  
                                                                
                                                                
       hub_id      dispatch_time delivery_completed_at  delivery_status
 Length   :950   Length   :950   Length   :950         Length   :950   
 N.unique :  8   N.unique :950   N.unique :932         N.unique :  3   
 N.blank  :  0   N.blank  :  0   N.blank  : 19         N.blank  :  0   
 Min.nchar:  3   Min.nchar: 19   Min.nchar:  0         Min.nchar:  6   
 Max.nchar:  3   Max.nchar: 19   Max.nchar: 26         Max.nchar:  7   
                                                

In [ ]:
# Quick test to confirm sqldf is working correctly
result <- sqldf("SELECT * FROM deliveries LIMIT 3")
print(result)


  delivery_id order_id driver_id vehicle_id hub_id       dispatch_time
1     DL00001   O00938      D004       V056    H05 2024-06-18 10:57:00
2     DL00002   O00004      D138       V007    H02 2025-01-11 18:45:00
3     DL00003   O00639      D006       V049    H02 2025-06-02 20:39:00
       delivery_completed_at delivery_status route_distance_km
1 2024-06-19 09:05:59.904311          Failed             17.26
2 2025-01-11 17:39:00.000000          OnTime             10.34
3 2025-06-02 21:45:32.366770          OnTime              7.92
  manual_route_override_count proof_of_completion_missing
1                           1                           0
2                           1                           0
3                           0                           0
  customer_rating_post_delivery fuel_or_charge_cost duration_hours
1                          3.07               12.05      22.149973
2                          5.00               13.41       1.100000
3                          4.98

## QUERY 1
The query below tells the total_deliveries and the failure_rate_pct for all the hubs in North Star. After running the query we find that Midtown Relay which is in 'Central zone' has the highest failure percentage which is consistent with our previous analysis.

---
Notably, the two worst performing hubs are both located in the Central zone, suggesting a zone-level operational issue rather than an isolated hub problem


In [ ]:
# Query 1 - Hub failure rates
# Optimisation: only selecting the columns needed rather than SELECT *
# JOIN is done on hub_id which is the primary key - efficient lookup
sql_query <- paste(
  "SELECT deliveries.hub_id, hub_name, zone, COUNT(delivery_id) as total_deliveries,",
  "ROUND(SUM(CASE WHEN delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(delivery_id), 2) AS failure_rate_pct",
  "FROM deliveries",
  "JOIN hubs ON deliveries.hub_id = hubs.hub_id",
  "GROUP BY deliveries.hub_id",
  "ORDER BY failure_rate_pct DESC",
  sep = "
"
)

result <- sqldf(sql_query)
print(result)


  hub_id       hub_name      zone total_deliveries failure_rate_pct
1    H08  Midtown Relay   Central              128            20.31
2    H05   Central Core   Central              115            20.00
3    H06    Airport Hub   Airport              104            14.42
4    H04      West Gate      West              127            12.60
5    H01 North Exchange     North              136            12.50
6    H07  Riverside Hub Riverside              115            12.17
7    H02     South Link     South              106             9.43
8    H03      East Dock      East              119             9.24


## QUERY 2
The below query tells delay and failure rates based on the service_type. After, running the query we find that the Retail service has the highest delay percentage. Also, Business sector has the highest failure rate.

This is concerning for North Star as Retail clients have strict windows and high expectations and Business contributes a good chunk to North Star.

In [ ]:
# Query 2 - Service type performance
# Optimisation: using CASE WHEN inside SUM instead of multiple subqueries
# This calculates both failure and delay rates in a single pass through the data
sql_query <- paste(
  "SELECT service_type,",
  "ROUND(SUM(CASE WHEN delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(delivery_id), 2) AS failure_rate_pct,",
  "ROUND(SUM(CASE WHEN delivery_status = 'Delayed' THEN 1 ELSE 0 END) * 100.0 / COUNT(delivery_id), 2) AS delay_rate_pct",
  "FROM orders",
  "JOIN deliveries ON orders.order_id = deliveries.order_id",
  "GROUP BY service_type",
  "ORDER BY delay_rate_pct DESC",
  sep="
"
)

result <- sqldf(sql_query)
print(result)


  service_type failure_rate_pct delay_rate_pct
1       Retail            12.50          22.32
2     Business            19.84          22.22
3       Parcel            10.87          21.30
4      Medical            14.81          20.37
5    Passenger            14.50          20.23


## QUERY 3
The below query tries to find the relation between driver overrides and the impact on the delivery. We find that, drivers with highest override count have the highest delay rate. This makes sense as changing the route continuously causes an impact to the delivery time.
Another interesting finding is that high override count doesn't directly correlate to highest failure rate.

In [ ]:
# Query 3 - Driver override analysis
# Optimisation: LIMIT 10 reduces the output to only the most relevant rows
# AVG is used instead of SUM to make override counts comparable across drivers with different delivery volumes
sql_query <- paste(
  "SELECT drivers.driver_id, driver_rating,",
  "AVG(manual_route_override_count) AS override_count,",
  "ROUND(SUM(CASE WHEN delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(delivery_id), 2) AS failure_rate_pct,",
  "ROUND(SUM(CASE WHEN delivery_status = 'Delayed' THEN 1 ELSE 0 END) * 100.0 / COUNT(delivery_id), 2) AS delay_rate_pct",
  "FROM drivers JOIN deliveries ON drivers.driver_id = deliveries.driver_id",
  "GROUP BY drivers.driver_id",
  "ORDER BY override_count DESC LIMIT 10"
)

result <- sqldf(sql_query)
print(result)


   driver_id driver_rating override_count failure_rate_pct delay_rate_pct
1       D112          4.51       4.500000             0.00         100.00
2       D127          4.19       2.833333             0.00          16.67
3       D021          4.24       2.500000             0.00          50.00
4       D139          4.99       2.000000             0.00          20.00
5       D130          3.64       2.000000            12.50           0.00
6       D124          3.78       2.000000             0.00          75.00
7       D105          3.71       2.000000            14.29          14.29
8       D085          4.11       2.000000             0.00          50.00
9       D079          4.48       2.000000             0.00          50.00
10      D069          5.00       2.000000            14.29          28.57


## QUERY 4
The query below checks the complaint count and type for each zone.
We find that Central zone has the highest number of complaints on different issues like Delay, Driver Behaviour, etc. This supports our earlier finding of issues in Midtown Relay. For North Star this means that the Central hub is not performing up to the standards.

In [ ]:
# Query 4 - Complaint analysis by zone (4 table join)
# Optimisation: LIMIT 15 avoids returning the full result set - we only need the top results
# JOIN order follows the natural key chain: complaints -> orders -> deliveries -> hubs
# This avoids unnecessary cross joins
sql_query <- paste(
  "SELECT zone, complaint_type, COUNT(complaint_id) AS complaint_count FROM complaints",
  "JOIN orders ON complaints.order_id = orders.order_id",
  "JOIN deliveries ON orders.order_id = deliveries.order_id",
  "JOIN hubs ON deliveries.hub_id = hubs.hub_id",
  "GROUP BY zone, complaint_type",
  "ORDER BY complaint_count DESC LIMIT 15"
)

result <- sqldf(sql_query)
print(result)


        zone    complaint_type complaint_count
1    Central             Delay              19
2    Central   DriverBehaviour              13
3      North             Delay              13
4    Central      MissedPickup              11
5    Central          AppIssue              10
6       East             Delay              10
7  Riverside             Delay              10
8  Riverside      MissedPickup              10
9       West             Delay              10
10   Central SupportExperience               9
11      East      MissedPickup               9
12 Riverside          AppIssue               8
13   Airport   DriverBehaviour               7
14   Airport             Delay               6
15      West          AppIssue               6


## QUERY 5
The query checks failure and delay rates by zone.

Central zone has the highest failure rate, consistent with findings in Query 1 and Query 4. Airport zone has the highest delay rate, suggesting a different type of operational problem. Deliveries are completing but taking longer than expected, possibly due to airport traffic or access restrictions.

These two zones need different interventions: Central for reducing outright failures, and Airport for improving delivery time.


In [ ]:
# Query 5 - Zone level delivery performance
# Optimisation: aggregating at zone level rather than hub level reduces the result set
# and makes the output easier to interpret for management
sql_query <- paste(
  "SELECT zone, ROUND(SUM(CASE WHEN delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(delivery_id), 2) AS failure_rate_pct,",
  "ROUND(SUM(CASE WHEN delivery_status = 'Delayed' THEN 1 ELSE 0 END) * 100.0 / COUNT(delivery_id), 2) AS delay_rate_pct",
  "FROM hubs JOIN deliveries ON hubs.hub_id = deliveries.hub_id",
  "GROUP BY zone",
  "ORDER BY failure_rate_pct DESC"
)

result <- sqldf(sql_query)
print(result)


       zone failure_rate_pct delay_rate_pct
1   Central            20.16          19.34
2   Airport            14.42          25.96
3      West            12.60          22.05
4     North            12.50          19.12
5 Riverside            12.17          21.74
6     South             9.43          24.53
7      East             9.24          19.33


## SUMMARY
The analysis reveals significant operational issues, particularly in the Central zone, which consistently shows the highest failure rates across hubs and a disproportionately high number of complaints (Query 1, 4, 5). The Airport zone also faces challenges, primarily with delivery delays (Query 5). Furthermore, Retail and Business service types exhibit concerning delay and failure rates, respectively (Query 2). Lastly, frequent manual route overrides by drivers are strongly correlated with increased delivery delays, indicating a potential area for optimization (Query 3).